La NASA Exoplanet Archive permite descargar curvas de luz de Kepler en formato CSV o FITS. Puedes automatizar la descarga de miles de estrellas (tanto con planetas confirmados como de control) usando astroquery o lightkurve, y luego aplicarles exactamente el mismo pipeline de preprocesamiento que ya tienes (clipping, normalización robusta y suavizado).

Los pasos serían:

Obtener la lista de estrellas: Usa el catálogo de KOI (Kepler Objects of Interest) para obtener los kepid de estrellas con planetas confirmados y de estrellas sin señal (falsos positivos o de control).

Descargar las curvas de luz: Con lightkurve, puedes descargar la curva de luz completa (todos los trimestres) de cada estrella y obtener un array de tiempo y flujo.

Procesar y alinear: Para cada estrella, aplicas tu pipeline (clipping P1-P99, normalización mediana/MAD, Savitzky-Golay). Luego, si la curva no tiene 3197 puntos, la interpolas o la recortas para que todas tengan la misma longitud.

Guardar en CSV: Construyes un DataFrame donde cada fila es una estrella, la primera columna es la etiqueta (LABEL) y las siguientes son los valores de flujo. Guardas como exoTrain_large.csv.

Ventajas:

Mismo formato: Tu Notebook 01 funcionaría sin cambios, solo cambiando la ruta de carga.

Control total: Puedes decidir cuántas estrellas incluir (miles, decenas de miles).

Clase minoritaria ampliada: Al incluir todas las estrellas con planetas confirmados del catálogo KOI (hay ~2,700), tendrías una clase positiva mucho más grande que los 37 actuales.

Preprocesamiento ya probado: Tu pipeline de clipping, normalización y suavizado está listo para aplicarse a estos datos.

Plan de Acción Inmediato
Si te parece bien este camino, puedo ayudarte a escribir el script de descarga y construcción del dataset. Empezaríamos con un subconjunto manejable (por ejemplo, 500 estrellas con planetas confirmados y 500 sin planeta) para probar que el pipeline funciona, y luego escalaríamos a miles.

# 1. Cargar catálogo KOI para obtener kepids de estrellas con planetas confirmados
# 2. Para cada kepid, descargar la curva de luz con lightkurve
# 3. Aplicar tu pipeline de preprocesamiento (clip_outliers, robust_normalize, smooth_curves)
# 4. Alinear a 3197 puntos
# 5. Construir DataFrame y guardar como CSV


Vamos a construir tu propio dataset de curvas de luz de Kepler, mucho más grande y manteniendo el formato que ya dominas. La estrategia es descargar el catálogo oficial de objetos de interés (KOI) de la NASA, obtener las curvas de luz con la librería lightkurve, y aplicarles tu pipeline de preprocesamiento para finalmente guardarlas en un CSV.

Aquí tienes el plan detallado, paso a paso.

📋 Paso 1: Preparar el Entorno
Primero, instala las librerías que necesitas. lightkurve es la herramienta estándar para acceder a los datos de Kepler, y astroquery te permitirá consultar el catálogo de la NASA.

In [ ]:
!pip install lightkurve astroquery

Descargar el Catálogo de Objetos de Interés (KOI)
El catálogo KOI es tu fuente de verdad. Contiene los kepid (identificadores de estrellas) y su clasificación (koi_disposition: CONFIRMED, CANDIDATE, FALSE POSITIVE).

Puedes descargarlo directamente desde el archivo de Exoplanetas de la NASA. El archivo CSV tiene ~9,564 filas y 140 columnas.


In [ ]:
# ============================================================
# 1 — Descarga del catalogo KOI desde NASA Exoplanet Archive
# ============================================================
# Este bloque obtiene la tabla "cumulative" del archivo KOI
# (Kepler Object of Interest) usando astroquery.
#
# La tabla "cumulative":
#   - Es el conjunto mas completo de KOIs publicado por Kepler.
#   - Incluye candidatos, confirmados y falsos positivos.
#   - Contiene parametros fisicos y geometricos de cada KOI.
#   - Se actualiza periodicamente por el NASA Exoplanet Archive.
#
# Campos seleccionados:
#   - kepid           : ID de la estrella en Kepler Input Catalog
#   - koi_disposition : estado del KOI (CONFIRMED, CANDIDATE, FALSE POSITIVE)
#   - koi_period      : periodo orbital estimado (dias)
#   - koi_time0bk     : tiempo central del transito (BKJD)
#   - koi_duration    : duracion del transito (horas)
#
# Esta informacion es util para:
#   - Analisis estadistico de KOIs reales
#   - Comparacion contra predicciones del modelo SNN
#   - Generacion de curvas sinteticas basadas en parametros reales
#   - Validacion externa del pipeline
#
# El resultado se convierte a un DataFrame de pandas para
# facilitar filtrado, analisis y visualizacion.

from astroquery.ipac.nexsci.nasa_exoplanet_archive import NasaExoplanetArchive
import pandas as pd

# ------------------------------------------------------------
# 1 — Consulta al archivo KOI
# ------------------------------------------------------------
koi_table = NasaExoplanetArchive.query_criteria(
    table="cumulative",
    select="kepid, koi_disposition, koi_period, koi_time0bk, koi_duration"
)

# ------------------------------------------------------------
# 2 — Convertir a DataFrame
# ------------------------------------------------------------
koi_df = koi_table.to_pandas()

# ------------------------------------------------------------
# 3 — Resumen del catalogo
# ------------------------------------------------------------
print(f"Total de KOIs en el catalogo: {len(koi_df)}")
print(koi_df['koi_disposition'].value_counts())


Total de KOIs en el catalogo: 9564
koi_disposition
FALSE POSITIVE    4839
CONFIRMED         2748
CANDIDATE         1977
Name: count, dtype: int64


Descargar las Curvas de Luz con lightkurve
Ahora, para cada kepid en tu lista, descargarás su curva de luz. La misión Kepler observó en "trimestres" (quarters), así que necesitas unir (stitch) los datos de todos los trimestres disponibles para obtener una serie temporal continua

In [2]:
# ============================================================
# 2 — Descarga y union ("stitch") de curvas de luz Kepler
# ============================================================
# Esta funcion obtiene todas las curvas de luz disponibles para un
# KEPID en la mision Kepler y las une en una sola serie continua.
#
# Flujo:
#   1) Buscar todas las curvas de luz asociadas al KEPID
#   2) Descargar cada trimestre (Quarter) como LightCurve
#   3) Unirlas con .stitch() para formar una curva completa
#
# Por que es importante:
#   - Kepler observa cada estrella en ~17 trimestres separados.
#   - Cada trimestre tiene su propio archivo FITS.
#   - Para analisis de transitos reales, es necesario unirlos.
#   - Lightkurve maneja automaticamente:
#         * gaps
#         * saltos de normalizacion
#         * concatenacion temporal
#
# La funcion devuelve:
#   - stitched_lc : objeto LightCurve con toda la serie temporal
#   - None        : si no hay datos o ocurre un error
#
# Esta funcion es ideal para:
#   - validar tu SNN con curvas reales
#   - extraer ventanas alrededor de transitos KOI
#   - generar datasets reales para test externo

import lightkurve as lk
import numpy as np

def get_stitched_lightcurve(kepid):
    """Descarga y une las curvas de luz de todos los trimestres de un kepid."""
    try:
        # ------------------------------------------------------------
        # 1 — Buscar todas las curvas de luz del KEPID
        # ------------------------------------------------------------
        search_result = lk.search_lightcurve(
            f"KIC {kepid}",
            author='Kepler',
            cadence='long'
        )
        if len(search_result) == 0:
            return None
        
        # ------------------------------------------------------------
        # 2 — Descargar todos los trimestres
        # ------------------------------------------------------------
        lc_collection = search_result.download_all()
        if lc_collection is None or len(lc_collection) == 0:
            return None
            
        # ------------------------------------------------------------
        # 3 — Unir todas las curvas en una sola serie continua
        # ------------------------------------------------------------
        stitched_lc = lc_collection.stitch()
        return stitched_lc

    except Exception as e:
        print(f"Error con kepid {kepid}: {e}")
        return None


d:\AI\Conectoma\exoplanetas-snn\.venv\Lib\site-packages\lightkurve\prf\__init__.py:7: UserWarning: Warning: the tpfmodel submodule is not available without oktopus installed, which requires a current version of autograd. See #1452 for details.
  warnings.warn(


lightkurve ofrece dos tipos de flujo: SAP y PDCSAP. Usa siempre PDCSAP, ya que está procesado específicamente para la detección de exoplanetas y elimina la mayoría de los efectos sistemáticos de la nave

Preprocesar cada Curva de Luz
Aquí aplicarás tu pipeline de preprocesamiento (clipping, normalización robusta, suavizado) a cada curva de luz descargada. lightkurve ya tiene métodos útiles como .normalize(), .flatten() y .remove_outliers() que puedes usar

In [3]:
# ============================================================
# 3 — Preprocesamiento de curvas de luz Kepler (pipeline real)
# ============================================================
# Esta funcion aplica un pipeline de preprocesamiento a una curva
# de luz real descargada con Lightkurve. El objetivo es producir
# una serie limpia, normalizada y comparable con las curvas del
# dataset usado para entrenar la SNN.
#
# Pasos aplicados:
#
#   1) remove_nans()
#      - Elimina puntos NaN del FITS original.
#      - Kepler contiene gaps y puntos corruptos por telemetria.
#
#   2) normalize()
#      - Normaliza el flujo dividiendo por la mediana.
#      - Equivalente a "flux / median(flux)".
#      - Facilita comparar curvas de distintas estrellas.
#
#   3) remove_outliers()
#      - Elimina picos extremos (cosmic rays, saltos instrumentales).
#      - sigma=20 elimina outliers muy severos.
#      - sigma_upper=4 elimina picos positivos fuertes.
#
#   4) flatten()
#      - Elimina tendencias de largo plazo (variabilidad estelar).
#      - Usa un filtro de ventana 401 (aprox. 8 horas).
#      - Mantiene la forma del transito sin deformarlo.
#
# Resultado:
#   - Una curva de luz limpia, normalizada y aplanada.
#   - Lista para extraer ventanas alrededor de transitos KOI.
#   - Compatible con tu pipeline de preprocesamiento para la SNN.
#
# Esta funcion es esencial para:
#   - Validacion externa con curvas reales.
#   - Comparar predicciones del modelo contra KOIs confirmados.
#   - Generar datasets realistas para test.
#   - Extraer segmentos de transito para analisis detallado.

def preprocess_lightcurve(lc):
    """Aplica el pipeline de preprocesamiento a una curva de luz."""
    
    # ------------------------------------------------------------
    # 1 — Eliminar valores NaN
    # ------------------------------------------------------------
    lc = lc.remove_nans()
    
    # ------------------------------------------------------------
    # 2 — Normalizar por la mediana
    # ------------------------------------------------------------
    lc = lc.normalize()
    
    # ------------------------------------------------------------
    # 3 — Eliminar outliers extremos (cosmic rays)
    # ------------------------------------------------------------
    lc = lc.remove_outliers(sigma=20, sigma_upper=4)
    
    # ------------------------------------------------------------
    # 4 — Aplanar tendencias de largo plazo
    # ------------------------------------------------------------
    lc = lc.flatten(window_length=401)
    
    return lc


Alinear las Curvas a una Longitud Fija
Tu SNN espera un vector de entrada de longitud fija (en tu caso, 3197 puntos). Como las curvas de luz de Kepler tienen duraciones diferentes, necesitas interpolarlas a una longitud común. El artículo de Shallue & Vanderburg (2018) sugiere "binning" (agrupar) las curvas a una longitud fija, como 2001 bins para la vista globa

In [4]:
# ============================================================
# 4 — Alineacion de curvas Kepler a longitud fija (3197 puntos)
# ============================================================
# Esta funcion toma una curva de luz real (LightCurve de Lightkurve)
# y la transforma a una longitud fija, compatible con el input de la
# FlyInspiredSNN.
#
# Problema:
#   - Las curvas Kepler tienen longitudes variables.
#   - Tu modelo SNN requiere exactamente 3197 puntos.
#
# Estrategia:
#   1) Intentar binning directo con lc.bin(bins=target_length)
#      - Lightkurve agrupa los puntos en bins uniformes.
#      - Es rapido y preserva la forma general.
#
#   2) Si el binning no produce exactamente target_length puntos:
#      - Interpolar manualmente con numpy.interp
#      - Se crea un eje normalizado [0,1] para reescalar la curva.
#
# Resultado:
#   - Un vector numpy de longitud fija (3197)
#   - Listo para pasar al modelo SNN
#
# Esta funcion es esencial para:
#   - Validacion externa con curvas reales
#   - Comparar KOIs confirmados contra predicciones del modelo
#   - Construir datasets reales con longitud uniforme
#   - Integrar curvas Kepler en tu pipeline sin modificar la arquitectura

def align_lightcurve(lc, target_length=3197):
    """Interpola la curva de luz a una longitud fija."""
    
    # ------------------------------------------------------------
    # 1 — Intentar binning directo
    # ------------------------------------------------------------
    lc_binned = lc.bin(bins=target_length)
    
    # ------------------------------------------------------------
    # 2 — Si el binning no produce la longitud deseada, interpolar
    # ------------------------------------------------------------
    if len(lc_binned.flux) != target_length:
        # Ejes normalizados para interpolacion
        x_old = np.linspace(0, 1, len(lc_binned.flux))
        x_new = np.linspace(0, 1, target_length)
        
        flux_interp = np.interp(x_new, x_old, lc_binned.flux)
        return flux_interp
    
    # ------------------------------------------------------------
    # 3 — Caso ideal: binning exacto
    # ------------------------------------------------------------
    return lc_binned.flux.value


Construir y Guardar el Dataset
Finalmente, itera sobre tu lista de kepid, descarga, preprocesa, alinea y guarda cada curva en una lista. Luego, construye un DataFrame donde cada fila sea una estrella y cada columna un punto de flujo.

In [5]:
# ============================================================
# 5 — Construccion de dataset real Kepler (KOIs confirmados y FP)
# ============================================================
# Esta funcion construye un dataset de curvas de luz reales
# descargadas desde la mision Kepler usando Lightkurve.
#
# Flujo completo:
#   1) Filtrar KOIs por disposicion (CONFIRMED vs FALSE POSITIVE)
#   2) Descargar todos los trimestres de cada estrella (get_stitched_lightcurve)
#   3) Preprocesar la curva (remove_nans, normalize, flatten, etc.)
#   4) Alinear la curva a longitud fija (3197 puntos)
#   5) Guardar:
#        - LABEL  (1=CONFIRMED, 0=FALSE POSITIVE)
#        - KEPID  (ID de la estrella)
#        - 3197 columnas de flux
#
# Objetivo:
#   - Crear un dataset realista para validar el modelo SNN
#   - Comparar predicciones contra KOIs confirmados
#   - Generar un conjunto de test externo completamente real
#
# max_stars:
#   - Limita el numero de estrellas procesadas
#   - Util para pruebas rapidas antes de construir el dataset completo

def build_dataset(koi_df, max_stars=1000):
    """Construye el dataset de curvas de luz alineadas."""
    data = []
    labels = []
    kepids = []
    
    # ------------------------------------------------------------
    # 1 — Filtrar solo CONFIRMED y FALSE POSITIVE
    # ------------------------------------------------------------
    filtered_df = koi_df[koi_df['koi_disposition'].isin(['CONFIRMED', 'FALSE POSITIVE'])]
    
    # ------------------------------------------------------------
    # 2 — Procesar cada estrella
    # ------------------------------------------------------------
    for idx, row in filtered_df.iterrows():
        if len(data) >= max_stars:
            break
            
        kepid = row['kepid']
        label = 1 if row['koi_disposition'] == 'CONFIRMED' else 0
        
        print(f"Procesando {len(data)+1}: KIC {kepid}...")
        
        # 2.1 — Descargar curva completa (todos los trimestres)
        lc = get_stitched_lightcurve(kepid)
        if lc is None:
            continue
            
        # 2.2 — Preprocesar curva (normalizar, flatten, etc.)
        lc_processed = preprocess_lightcurve(lc)
        
        # 2.3 — Alinear a longitud fija (3197 puntos)
        flux_aligned = align_lightcurve(lc_processed)
        
        # 2.4 — Guardar si la curva es valida
        if flux_aligned is not None:
            data.append(flux_aligned)
            labels.append(label)
            kepids.append(kepid)
    
    # ------------------------------------------------------------
    # 3 — Construir DataFrame final
    # ------------------------------------------------------------
    df = pd.DataFrame(data)
    df.insert(0, 'LABEL', labels)
    df.insert(1, 'KEPID', kepids)
    
    return df

# ------------------------------------------------------------
# 4 — Construir dataset de prueba (100 estrellas)
# ------------------------------------------------------------
dataset_df = build_dataset(koi_df, max_stars=100)
print(f"Dataset construido: {dataset_df.shape}")

# ------------------------------------------------------------
# 5 — Guardar dataset en CSV
# ------------------------------------------------------------
dataset_df.to_csv('../data/kepler_dataset_large.csv', index=False)
print("Dataset guardado en ../data/kepler_dataset_large.csv")


Procesando 1: KIC 10797460...
Procesando 2: KIC 10797460...
Procesando 3: KIC 10848459...
Procesando 4: KIC 10854555...
Procesando 5: KIC 10872983...
Procesando 6: KIC 10872983...
Procesando 7: KIC 10872983...
Procesando 8: KIC 6721123...
Procesando 9: KIC 10910878...
Procesando 10: KIC 11446443...
Procesando 11: KIC 10666592...
Procesando 12: KIC 6922244...
Procesando 13: KIC 10984090...
Procesando 14: KIC 10419211...
Procesando 15: KIC 10464078...
Procesando 16: KIC 10480982...
Procesando 17: KIC 10485250...
Procesando 18: KIC 10526549...
Procesando 19: KIC 10583066...
Procesando 20: KIC 10583180...
Procesando 21: KIC 10601284...
Procesando 22: KIC 10601284...
Procesando 23: KIC 10601284...
Procesando 24: KIC 2306756...
Procesando 25: KIC 10662202...
Procesando 26: KIC 10682541...
Procesando 27: KIC 11460018...
Procesando 28: KIC 11463211...
Procesando 29: KIC 11465813...
Procesando 30: KIC 11493732...
Procesando 31: KIC 11507101...
Procesando 32: KIC 10910878...
Procesando 33: KIC 1